# Fine-tune FinBERT on Financial PhraseBank

Grid search over `learning_rate × num_epochs` to identify the best FinBERT configuration. Selection criterion: **best validation macro-F1**.

Default grid:
- `learning_rate ∈ {1e-5, 2e-5, 5e-5}`
- `num_epochs ∈ {3, 4}`
- `batch_size = 16` (fixed, matches proposal)

= 6 configurations. **Total runtime: ~50-70 minutes on a Colab T4 GPU.** Each run saves its own predictions file to `predictions/sweep/` for inspection. After the sweep, the best config is retrained once and saved as the canonical `finbert_finetuned_predictions.csv`.

Why this grid is defensible: learning rate dominates BERT fine-tuning outcomes (Mosbach et al., 2021). Sweeping `lr` across an order of magnitude covers the commonly successful range. Epochs at 3 and 4 guard against over/underfitting on ~3,900 training sentences.

## Setup

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('financial-sentiment-comparison'):
        !git clone https://github.com/maximusrome/financial-sentiment-comparison.git
    %cd financial-sentiment-comparison
    !pip install -q transformers==4.44.2 pandas scikit-learn pyarrow matplotlib seaborn

from pathlib import Path
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

In [ ]:
from data_loader import build_and_save_split, load_split, SPLIT_PATH
if not SPLIT_PATH.exists():
    build_and_save_split()

## Run the sweep

In [ ]:
from finbert_train import run_sweep

sweep_df = run_sweep(
    model_name='ProsusAI/finbert',
    output_name='finbert_finetuned',
    learning_rates=[1e-5, 2e-5, 5e-5],
    num_epochs_list=[3, 4],
    batch_size=16,
    seed=42,
)

## Sweep results table

In [ ]:
import pandas as pd
sweep_df = pd.read_csv('results/tables/finbert_sweep.csv')
sweep_df.round(4)

## Visualize the sweep

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(6, 4))
for n_ep, group in sweep_df.groupby('num_epochs'):
    g = group.sort_values('learning_rate')
    ax.plot(g['learning_rate'], g['val_f1'], marker='o',
            label=f'{n_ep} epochs')
ax.set_xscale('log')
ax.set_xlabel('Learning rate (log scale)')
ax.set_ylabel('Validation macro-F1')
ax.set_title('FinBERT hyperparameter sweep')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/figures/finbert_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## Final evaluation (best config retrained)

`run_sweep` already retrained the best config and saved it to `predictions/finbert_finetuned_predictions.csv`. Evaluate with the standard framework:

In [ ]:
from evaluation import evaluate_predictions
result = evaluate_predictions('predictions/finbert_finetuned_predictions.csv')
print(f"Test accuracy:  {result['overall']['accuracy']:.4f}")
print(f"Test macro-F1:  {result['overall']['macro_f1']:.4f}\n")
print('By agreement tier:')
for tier, m in result['by_tier'].items():
    print(f"  tier={tier:6s}  acc={m['accuracy']:.4f}  macro_f1={m['macro_f1']:.4f}  (n={m['n']})")

In [ ]:
!zip -r /content/output.zip data predictions results
from google.colab import files
files.download('/content/output.zip')